# 05-Counterfactual Reasoning (Alibi Explain)

In the previous modules, we deployed SHAP, LIME, and Integrated Gradients to explain exactly *why* a model made a specific prediction. However, if a Deep Learning model denies a user a mortgage, handing them a SHAP waterfall plot that says, *"You were denied because your Debt-to-Income ratio has a Shapley value of -2.4"* is legally compliant, but operationally useless to the user.

To achieve true **Algorithmic Recourse**, we must answer a different mathematical question: *"What is the absolute minimum change this specific user must make to their profile to flip the model's decision from Denied to Approved?"*

To engineer this, we transition from attribution to **Counterfactual Reasoning**. We will utilize **Alibi Explain**, deploying optimization algorithms to dynamically perturb the user's data until it crosses the decision boundary, while strictly enforcing mathematical sparsity so the suggested changes are realistic and actionable.

## 1. The Mathematics of Counterfactual Optimization

### The Recourse Objective Function

A counterfactual explanation seeks a new feature vector $x'$ (the counterfactual) that is as close as possible to the original input $x$, but yields the desired target prediction $y_{target}$ from the model $f$.

This is framed as an optimization problem where we minimize a composite loss function:


$$\mathcal{L}(x', x, y_{target}) = \lambda \cdot \ell(f(x'), y_{target}) + d(x, x')$$

* $\ell(f(x'), y_{target})$: The **Prediction Loss**. It penalizes the optimizer if the new point $x'$ does not result in the desired class (e.g., Mortgage Approved).
* $d(x, x')$: The **Distance Metric**. It penalizes the optimizer for moving $x'$ too far away from the original reality $x$.
* $\lambda$: A dynamic hyperparameter that balances the trade-off. If $\lambda$ is too high, the optimizer jumps to the target class but changes every feature drastically.

### Enforcing Actionable Sparsity (Elastic Net Regularization)

A good counterfactual must be **Sparse**. Telling a user to change 15 different variables is unactionable. Telling them to change just *one* variable (e.g., "Pay down $500 of credit card debt") is highly actionable.

Alibi enforces sparsity by defining the distance metric $d(x, x')$ using an **Elastic Net** combination of $L_1$ and $L_2$ norms:


$$d(x, x') = \beta_1 \vert{}\vert{}x - x'\vert{}\vert{}_1 + \beta_2 \vert{}\vert{}x - x'\vert{}\vert{}_2^2$$


The $L_1$ norm (Manhattan distance) mathematically forces the gradients of non-critical features to exactly zero, guaranteeing that the optimizer alters the absolute minimum number of features required to cross the decision boundary.

### Immutability Constraints

Not all features can be changed. A user cannot change their `Age`, `Past Default History`, or `Race`. In a production counterfactual engine, we mathematically freeze these axes during gradient descent:


$$\frac{\partial \mathcal{L}}{\partial x'_{immutable}} \equiv 0$$

## 2. Imperative Execution: The Counterfactual Engine (Alibi + Keras)

Let's build a local, zero-cost sandbox. We will train a simple Keras Neural Network on a simulated credit dataset. Then, we will deploy Alibi's `Counterfactual` explainer to find the minimal mathematical path to turn a Loan Rejection into a Loan Approval.

In [2]:
# Required installations: pip install alibi tensorflow scikit-learn numpy pandas
import numpy as np
import tensorflow as tf
from tensorflow import keras
from alibi.explainers import Counterfactual
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print("--- 🚀 Initializing Alibi Counterfactual Recourse Sandbox ---")

# 1. 📝 Simulate a Credit Approval Dataset
# Features: [Income, Credit Score, Debt, Age]
np.random.seed(42)
X = np.random.rand(1000, 4) * [100000, 800, 50000, 60] + [20000, 300, 0, 18]
# Synthetic logic: High income, high credit, low debt = Approved (1)
y = ((X[:, 0] > 60000) & (X[:, 1] > 650) & (X[:, 2] < 20000)).astype(int)

# Normalize the data (Neural Networks and Distance Metrics require scaled inputs)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 2. 🧠 Train a Black-Box Neural Network
model = keras.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=(4,)),
    keras.layers.Dense(8, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid') # Output: Probability of Approval
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

# Isolate a specific denied applicant (e.g., Index 5)
denied_idx = 5
X_instance_scaled = X_test[denied_idx:denied_idx+1]
original_prediction = model.predict(X_instance_scaled, verbose=0)[0][0]

print(f"   [SYSTEM] Target Applicant isolated.")
print(f"   [SYSTEM] Original Neural Network Probability: {original_prediction:.2%} (REJECTED)")

print("\n--- 🧮 Executing Counterfactual Gradient Descent ---")

# 3. ⚡ Instantiate the Alibi Counterfactual Explainer
# We target class 1.0 (Approved). We enforce a specific learning rate and distance metric.
cf_explainer = Counterfactual(
    predict_fn=model,
    shape=(1, 4),
    target_proba=0.80, # We want to push the probability comfortably past the 0.5 threshold
    tol=0.01,
    target_class='other',
    max_iter=1000,
    learning_rate_init=0.1,
    feature_range=(X_train.min(axis=0), X_train.max(axis=0)) # Ensure CF stays in realistic bounds
)

# 4. Fit the explainer to the specific denied instance
explanation = cf_explainer.explain(X_instance_scaled)

print("   ✅ Counterfactual Optimization Complete. Recourse path found.")

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 3. Declarative Telemetry: Extracting the Recourse Delta ($\Delta$)

The raw output of the counterfactual optimizer is a scaled tensor. We must inverse-transform this tensor back into human-readable business logic (dollars and points) and compute the exact **Delta** required for algorithmic recourse.

In [ ]:
import pandas as pd

print("\n--- 🔍 Auditing the Minimal Recourse Path ---")

if explanation.cf is not None:
    # 1. Extract the scaled counterfactual and inverse-transform to original units
    cf_scaled = explanation.cf['X']
    X_original_real = scaler.inverse_transform(X_instance_scaled)[0]
    X_cf_real = scaler.inverse_transform(cf_scaled)[0]
    
    # 2. Calculate the Actionable Delta
    feature_names = ['Income ($)', 'Credit Score', 'Debt ($)', 'Age']
    delta = X_cf_real - X_original_real
    
    # Create a telemetry dataframe for operational visibility
    recourse_df = pd.DataFrame({
        'Feature': feature_names,
        'Original Reality': X_original_real,
        'Required Counterfactual': X_cf_real,
        'Actionable Delta (Δ)': delta
    })
    
    # Verify the new prediction
    new_prediction = model.predict(cf_scaled, verbose=0)[0][0]
    
    print(f"   [New Counterfactual Probability]: {new_prediction:.2%} (APPROVED)\n")
    
    # Print formatted Recourse matrix
    for _, row in recourse_df.iterrows():
        # Mask microscopic changes enforced by floating-point math
        if abs(row['Actionable Delta (Δ)']) < 1.0:
            continue 
        
        direction = "INCREASE" if row['Actionable Delta (Δ)'] > 0 else "DECREASE"
        print(f"      * {direction} {row['Feature']:<14} by {abs(row['Actionable Delta (Δ)']):7.0f} (From {row['Original Reality']:7.0f} to {row['Required Counterfactual']:7.0f})")
    
    print("\n   -> Physics: Notice the sparsity. The algorithm did not tell the user to age 5 years or double their income. It identified the mathematical path of least resistance (e.g., paying off a specific amount of debt) to instantly flip the neural network's activation thresholds.")

else:
    print("   ❌ No valid counterfactual found within the specified tolerance and feature bounds.")

## 4. Visualizing Counterfactual Physics (The Decision Boundary Trajectory)

To mathematically comprehend *how* the optimizer finds this path, we must visualize the neural network's decision boundary in 2D space. We will plot the Original Instance (Rejected) and trace the $L_1$-regularized path to the Counterfactual Instance (Approved).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(12, 7))
fig.suptitle("Systems Architecture: Counterfactual Gradient Descent Trajectory", fontsize=16, fontweight='bold', color="#2c3e50")

# 1. Simulate the Neural Network Decision Boundary
# X-axis: Credit Score, Y-axis: Debt
x_credit = np.linspace(500, 800, 100)
y_debt = np.linspace(0, 50000, 100)
X_grid, Y_grid = np.meshgrid(x_credit, y_debt)

# Synthetic non-linear probability landscape
Z_prob = 1 / (1 + np.exp(-((X_grid - 650)/30 - (Y_grid - 20000)/10000)))

# Plot the probability contour
contour = ax.contourf(X_grid, Y_grid, Z_prob, levels=10, cmap='RdYlGn', alpha=0.6)
ax.contour(X_grid, Y_grid, Z_prob, levels=[0.5], colors='black', linewidths=3, linestyles='solid')
ax.text(700, 35000, "Decision Boundary (P=0.5)", ha='center', fontweight='bold', color='black', rotation=30)

# 2. Plot the Original Instance (Rejected Zone - Red)
orig_credit, orig_debt = 620, 30000
ax.scatter(orig_credit, orig_debt, color='darkred', s=300, marker='X', edgecolors='white', zorder=5, label='Original Reality $x$ (Rejected)')

# 3. Plot the Counterfactual Instance (Approved Zone - Green)
# Notice the L1 Sparsity: Only Credit Score moves, Debt stays exactly the same.
cf_credit, cf_debt = 710, 30000
ax.scatter(cf_credit, cf_debt, color='darkgreen', s=300, marker='o', edgecolors='white', zorder=5, label='Counterfactual $x\'$ (Approved)')

# 4. Draw the Minimal Actionable Path (The Delta)
ax.annotate("", xy=(cf_credit, cf_debt), xytext=(orig_credit, orig_debt), 
            arrowprops=dict(arrowstyle="->", lw=4, color="black", linestyle="--"))

# Formatting
ax.set_xlabel("Credit Score", fontsize=12, fontweight='bold')
ax.set_ylabel("Debt ($)", fontsize=12, fontweight='bold')
ax.legend(loc='lower left', frameon=True, shadow=True, fontsize=11)

# Annotate Physics
ax.annotate('L1-Regularized Path:\nThe optimizer mathematically refuses to move diagonally.\nMoving diagonally would require changing 2 variables.\nIt locks the Debt axis and forces 100% of the\ngradient update onto the Credit Score axis.', 
            xy=(665, 30000), xytext=(550, 10000),
            arrowprops=dict(facecolor='black', shrink=0.05, width=2, headwidth=8),
            ha='left', fontweight='bold', color='#2c3e50',
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#2c3e50", lw=1.5))

plt.tight_layout()
plt.show()

*(Insight: In an unregularized environment, the optimizer would shoot a straight, diagonal line directly toward the deep green area, telling the user to simultaneously alter their debt, credit score, age, and income by microscopic, impossible amounts. By applying $L_1$ Elastic Net regularization, the vector is forced to snap to the grid axes, delivering a singular, highly actionable piece of advice: "Increase your credit score by 90 points, and do absolutely nothing else.")*

## Real-World Use Case or Analogy

Think of SHAP vs. Counterfactual Reasoning like **Failing a Car Emission Test**:

* **SHAP (The Mechanic's Diagnosis):** You fail the smog check. You ask the mechanic why. The mechanic hands you a SHAP waterfall plot that says: *"Your catalytic converter is operating at 40% efficiency, your oxygen sensor voltage is off by 0.2V, and your engine RPM is fluctuating by 3%."* This tells you exactly *why* the machine failed you, but you have no idea what to physically do about it.
* **Counterfactual Reasoning (The Actionable Recourse):** Instead of explaining the engine physics, the mechanic's computer runs an optimization algorithm. It calculates millions of potential states and tells you: *"If you replace this specific $50 oxygen sensor, your car will pass the test."*

GDPR and modern AI ethics laws increasingly recognize that telling a user *why* an algorithm punished them is insufficient. Responsible AI requires handing the user the exact blueprint to change their outcome. Counterfactual optimization engines mathematically guarantee that this blueprint is both valid and achievable.